# 🔬 Breast Cancer Predictor — Clinical ML Pipeline
**Wisconsin Breast Cancer Diagnostic (WBCD) Dataset**  
*Wolberg et al. 1995 — UCI ML Repository ID 17*

---
This notebook walks through the complete 10-step pipeline:
1. Data Ingestion & Understanding
2. Preprocessing Pipeline
3. Feature Engineering & Selection
4. Class Imbalance Handling
5. Model Training & Hyperparameter Tuning
6. Evaluation Metrics — Clinical Priority
7. Explainability (Clinician-Facing)
8. Model Serialisation
9. Monitoring Strategy
10. Summary & Next Steps

> **Clinical note:** This system prioritises **minimising false negatives** (missed cancers).  
> Target sensitivity ≥ 0.97 · Threshold set via Youden Index

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, '..')   # allow `import src` from notebooks/
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Pretty display
pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'sans-serif'})

# Load config
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('✓ Environment ready  |  Python', sys.version.split()[0])
print('Config:', cfg['project']['name'], cfg['project']['version'])

## 1 — Data Ingestion & Understanding

In [ ]:
from src.data.loader import load_wbcd, split_data, feature_names

df = load_wbcd(raw_dir='../data/raw')
print(f'Dataset shape: {df.shape}  (569 samples, 30 features + target)')
print(f'\nClass distribution:')
counts = df['diagnosis'].value_counts().rename({0: 'Malignant', 1: 'Benign'})
print(counts)
print(f'\nClass ratio (benign/malignant): {counts["Benign"]/counts["Malignant"]:.2f}')

In [ ]:
# Quick look at the data
df.head(3)

In [ ]:
# Missing value check
missing = df.isnull().sum()
print('Missing values:', missing[missing > 0].to_dict() or 'None 🎉')
print('\nData types:')
print(df.dtypes.value_counts())

In [ ]:
# Descriptive statistics by class
from src.data.eda import print_summary
summary = print_summary(df, target_col='diagnosis')
# Show mean row for readability
summary.loc[summary.index.get_level_values(1) == 'mean'].head(10)

## 2 — Exploratory Data Analysis

In [ ]:
from src.data.eda import (
    plot_class_distribution,
    plot_feature_histograms,
    plot_pairplot,
    plot_correlation_heatmap,
)
import os; os.makedirs('../reports', exist_ok=True)

_ = plot_class_distribution(df, save_dir='../reports')
plt.show()

In [ ]:
_ = plot_feature_histograms(df, save_dir='../reports')
plt.show()

In [ ]:
_ = plot_pairplot(df, n_top=6, save_dir='../reports')
plt.show()

In [ ]:
_ = plot_correlation_heatmap(df, save_dir='../reports')
plt.show()

## 3 — Preprocessing Pipeline

In [ ]:
from src.data.preprocessor import build_preprocessing_pipeline

# Stratified split first
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df, target_col='diagnosis',
    test_size=cfg['data']['test_size'],
    val_size=cfg['data']['val_size'],
    random_state=RANDOM_STATE,
)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

# Build and fit pipeline
preprocessor = build_preprocessing_pipeline(
    correlation_threshold=cfg['preprocessing']['correlation_threshold']
)
X_train_pp = preprocessor.fit_transform(X_train, y_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

print(f'\nAfter preprocessing  → train: {X_train_pp.shape}  val: {X_val_pp.shape}  test: {X_test_pp.shape}')

# Which features were dropped?
dropper = preprocessor.named_steps['corr_dropper']
feat_names = list(X_train.columns)
dropped = [feat_names[i] for i in dropper.drop_cols_]
print(f'\nDropped (corr > {cfg["preprocessing"]["correlation_threshold"]}): {dropped}')

## 4 — Feature Engineering: SelectKBest vs PCA

In [ ]:
from src.data.preprocessor import select_k_best, apply_pca

k = cfg['preprocessing']['n_features_kbest']
X_tr_kb, X_test_kb, kb_selector, selected_feats = select_k_best(
    X_train_pp, y_train.values, X_test_pp, k=k, feature_names=feat_names
)
print(f'SelectKBest ({k} features): {selected_feats}')

# Feature scores
scores_df = pd.DataFrame({
    'feature': feat_names,
    'f_score': kb_selector.scores_,
    'p_value': kb_selector.pvalues_,
    'selected': kb_selector.get_support()
}).sort_values('f_score', ascending=False)
print('\nTop 10 features by F-score:')
scores_df.head(10)

In [ ]:
X_tr_pca, X_test_pca, pca = apply_pca(
    X_train_pp, X_test_pp,
    variance_threshold=cfg['preprocessing']['pca_variance']
)
print(f'PCA: {pca.n_components_} components retained ({cfg["preprocessing"]["pca_variance"]*100:.0f}% variance)')

# Plot explained variance
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, pca.n_components_ + 1), pca.explained_variance_ratio_, color='#5CC8C8', alpha=0.85)
ax.plot(range(1, pca.n_components_ + 1),
        np.cumsum(pca.explained_variance_ratio_), 'r-o', ms=4, label='Cumulative')
ax.axhline(cfg['preprocessing']['pca_variance'], ls='--', color='gray', lw=1.2,
           label=f"{cfg['preprocessing']['pca_variance']*100:.0f}% threshold")
ax.set_xlabel('Principal Component'); ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA — Explained Variance per Component', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## 5 — Model Training & Hyperparameter Tuning

In [ ]:
from src.models.trainer import train_all_models, summarise_cv_results

# Use class_weight='balanced' for all models; scale_pos_weight for XGBoost
scale_pos_weight = (y_train == 1).sum() / max((y_train == 0).sum(), 1)
print(f'scale_pos_weight for XGBoost: {scale_pos_weight:.3f}')
print(f'class_weight="balanced" for LR, RF, SVM, MLP')
print(f'SMOTE: {False}  (class_weight preferred for n=569)')
print('\nTraining … (this may take 5–15 min on CPU)')

In [ ]:
cv_results = train_all_models(
    X_train_pp, y_train.values,
    scale_pos_weight=float(scale_pos_weight),
    use_smote=False,
    cv_splits=cfg['cv']['n_splits'],
    scoring=cfg['cv']['scoring'],
)
cv_summary = summarise_cv_results(cv_results)
cv_summary

## 6 — Clinical Evaluation

In [ ]:
from src.evaluate.metrics import (
    evaluate_all, plot_roc_curves, plot_precision_recall_curves,
    plot_calibration_curves, plot_confusion_matrix, select_best_model
)

best_estimators = {name: gs.best_estimator_ for name, gs in cv_results.items()}

eval_df = evaluate_all(best_estimators, X_test_pp, y_test.values, use_youden=True)
eval_df

In [ ]:
_ = plot_roc_curves(best_estimators, X_test_pp, y_test.values, save_dir='../reports')
plt.show()

In [ ]:
_ = plot_precision_recall_curves(best_estimators, X_test_pp, y_test.values, save_dir='../reports')
plt.show()

_ = plot_calibration_curves(best_estimators, X_test_pp, y_test.values, save_dir='../reports')
plt.show()

In [ ]:
best_name = select_best_model(eval_df, target_sensitivity=cfg['clinical']['target_sensitivity'])
best_model = best_estimators[best_name]
print(f'\n✓  Best model: {best_name.upper()}')
print(eval_df[eval_df['model'] == best_name].T)

In [ ]:
_ = plot_confusion_matrix(best_model, X_test_pp, y_test.values,
                          model_name=best_name, save_dir='../reports')
plt.show()

## 7 — SHAP Explainability

In [ ]:
from src.evaluate.explainer import (
    build_shap_explainer, compute_shap_values,
    plot_feature_importance_bar, plot_beeswarm, plot_waterfall,
    top_features_by_shap, lime_explanation
)
import os; os.makedirs('../reports/shap_plots', exist_ok=True)

# Background dataset for KernelExplainer
n_bg = min(100, len(X_train_pp))
bg_idx = np.random.choice(len(X_train_pp), n_bg, replace=False)
X_background = X_train_pp[bg_idx]

shap_explainer = build_shap_explainer(best_model, X_background, model_name=best_name)
shap_values    = compute_shap_values(shap_explainer, X_test_pp, model_name=best_name)

# Resolve preprocessed feature names
proc_feats = preprocessor.named_steps['corr_dropper'].get_feature_names_out(feat_names)
if proc_feats is None: proc_feats = feat_names
proc_feats = list(proc_feats)
print(f'Feature count after preprocessing: {len(proc_feats)}')

In [ ]:
# Global importance bar
_ = plot_feature_importance_bar(shap_values, proc_feats, model_name=best_name,
                                save_dir='../reports/shap_plots')
plt.show()

In [ ]:
# Beeswarm / summary
_ = plot_beeswarm(shap_values, proc_feats, model_name=best_name,
                  max_display=cfg['shap']['beeswarm_max_display'],
                  save_dir='../reports/shap_plots')
plt.show()

In [ ]:
# Waterfall — explain a single patient's prediction
SAMPLE_IDX = 0   # Change to inspect other patients
_ = plot_waterfall(shap_explainer, X_test_pp, proc_feats,
                   sample_idx=SAMPLE_IDX, model_name=best_name,
                   save_dir='../reports/shap_plots')
plt.show()

true_label = 'Malignant' if y_test.values[SAMPLE_IDX] == 0 else 'Benign'
print(f'Patient #{SAMPLE_IDX}  |  True label: {true_label}')

In [ ]:
# Top features report
top_feats = top_features_by_shap(shap_values, proc_feats, n=10)
top_feats_df = pd.DataFrame(top_feats)
print('\nTop 10 Predictive Features (SHAP):')
print(top_feats_df.to_string(index=False))

In [ ]:
# LIME (secondary explainer)
lime_exp = lime_explanation(
    best_model, X_train_pp, X_test_pp,
    feature_names=proc_feats,
    class_names=['Malignant', 'Benign'],
    sample_idx=SAMPLE_IDX,
    save_dir='../reports/shap_plots'
)
if lime_exp:
    lime_exp.show_in_notebook(show_table=True, show_all=False)

## 8 — Serialise Best Model

In [ ]:
from src.models.trainer import save_model
import os; os.makedirs('../models/registry', exist_ok=True)

save_model(best_model,   '../models/registry/best_model.joblib')
save_model(preprocessor, '../models/registry/preprocessor.joblib')
save_model(shap_explainer, '../models/registry/shap_explainer.joblib', compress=1)
print('✓ All artefacts serialised')

## 9 — Monitoring & Drift Detection Demo

In [ ]:
from src.api.monitoring import DriftMonitor, RetrainingGate, PredictionLogger

# Save reference data
import os; os.makedirs('../data/processed', exist_ok=True)
ref_df = pd.DataFrame(X_train_pp)
ref_df.to_csv('../data/processed/reference_data.csv', index=False)

# Simulate drift monitor on test data
ref_pd = pd.DataFrame(X_train_pp, columns=[str(i) for i in range(X_train_pp.shape[1])])
cur_pd = pd.DataFrame(X_test_pp,  columns=[str(i) for i in range(X_test_pp.shape[1])])

monitor = DriftMonitor(
    reference_data=ref_pd,
    feature_names=[str(i) for i in range(X_train_pp.shape[1])],
)
report = monitor.run(cur_pd)
drifted = monitor.is_drift_detected(report)
print(f'Drift detected on test data: {drifted}')

# Retraining gate
gate = RetrainingGate()
best_row = eval_df[eval_df['model'] == best_name].iloc[0]
should, reason = gate.should_retrain(rolling_auc=best_row['auc_roc'], drift_detected=drifted)
print(f'Retraining needed: {should}  |  Reason: {reason}')

## 10 — Summary & Model Card

| Attribute | Value |
|-----------|-------|
| **Dataset** | Wisconsin Breast Cancer Diagnostic (Wolberg et al. 1995) |
| **Task** | Binary classification: malignant vs benign |
| **Samples** | 569 (357 benign / 212 malignant) |
| **Features** | 30 FNA nuclear features (mean, SE, worst) |
| **Split** | Stratified 70/15/15 |
| **Imbalance strategy** | class_weight='balanced' + Youden threshold |
| **CV** | Stratified 5-fold GridSearchCV |
| **Primary metric** | AUC-ROC |
| **Clinical constraint** | Sensitivity ≥ 0.97 |
| **Threshold** | Youden Index |
| **Explainability** | SHAP (global + local) + LIME |
| **Deployment** | FastAPI `/predict` + `/health` |
| **Monitoring** | Evidently drift + rolling AUC gate |
| **Seed** | 42 |

### ⚠️ Model Card Limitations (FDA/CE Guidelines)
- Trained on a single institution dataset; generalisation to other scanners/labs is unvalidated
- Should be used as **clinical decision support**, not as a standalone diagnostic tool
- Requires regular revalidation on prospective data
- False negative rate must be monitored continuously in production

In [ ]:
print('=' * 60)
print('FINAL EVALUATION SUMMARY')
print('=' * 60)
print(eval_df[['model','auc_roc','auc_pr','sensitivity','specificity','f1_macro','FN']]
      .to_string(index=False))
print()
print(f'✓ Selected model: {best_name.upper()}')
print(f'✓ Reports saved to: ../reports/')
print(f'✓ SHAP plots saved to: ../reports/shap_plots/')
print(f'✓ Model artefacts:  ../models/registry/')